In [ ]:
conda install -c conda-forge requests beautifulsoup4 jupyter

In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = requests.get(url)
response.encoding = 'utf-8'   # <-- add this line
print(response.status_code)

200


In [15]:
#find all books on the page:
soup = BeautifulSoup(response.text, "html.parser")
books = soup.find_all("article", class_="product_pod")
print(len(books))

20


In [16]:
# Extract and display the first book to verify the HTML was parsed correctly
first_book = books[0]
print(first_book.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [17]:
#extract the title, price, and rating from that first book:
# Title
title = first_book.h3.a["title"]

# Price (as text, includes £ symbol)
price_text = first_book.find("p", class_="price_color").text

# Rating (it's stored in the class name, like "star-rating Three")
rating_class = first_book.find("p", class_="star-rating")["class"]
rating = rating_class[1]  # this gets "Three", "Two", etc.

print("Title:", title)
print("Price:", price_text)
print("Rating:", rating)

Title: A Light in the Attic
Price: £51.77
Rating: Three


In [18]:
#convert price to INR:
# Remove the £ symbol and convert to a number
price_gbp = float(price_text.replace("£", ""))

# Conversion rate: 1 GBP ≈ 105 INR (approx, as of recent rates)
conversion_rate = 105

price_inr = price_gbp * conversion_rate

print("Price in GBP: £", price_gbp)
print("Price in INR: ₹", round(price_inr, 2))

Price in GBP: £ 51.77
Price in INR: ₹ 5435.85


In [19]:
#loop through all books and collect the data:
# Get the live exchange rate (optional, more accurate)
conversion_rate = 105  # approx GBP to INR

all_books = []  # empty list to store each book's data

for book in books:
    title = book.h3.a["title"]
    price_text = book.find("p", class_="price_color").text
    price_gbp = float(price_text.replace("£", ""))
    price_inr = round(price_gbp * conversion_rate, 2)
    rating_class = book.find("p", class_="star-rating")["class"]
    rating = rating_class[1]
    availability = book.find("p", class_="instock availability").text.strip()

    all_books.append({
        "Title": title,
        "Price (GBP)": price_gbp,
        "Price (INR)": price_inr,
        "Rating": rating,
        "Availability": availability
    })

print(len(all_books), "books collected")
print(all_books[0])  # show the first one as a sample

20 books collected
{'Title': 'A Light in the Attic', 'Price (GBP)': 51.77, 'Price (INR)': 5435.85, 'Rating': 'Three', 'Availability': 'In stock'}


In [20]:
#Convert Rating text → number
# Mapping word ratings to numbers
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# test it on the rating we already have
print(rating_map["Three"]) 

3


In [21]:
#Extract stock count (not just "In stock")
import re  # regex library, built into Python

sample_text = "In stock (19 available)"
match = re.search(r'\((\d+) available\)', sample_text)
stock_count = int(match.group(1))
print(stock_count)  

19


In [22]:
#test this on just ONE book first:
# Get the link for the first book
book_link = first_book.h3.a["href"]
full_link = "https://books.toscrape.com/" + book_link
print(full_link)

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [23]:
# Fetch the individual book's page (needed to get its category, which isn't on the homepage)
book_response = requests.get(full_link)
book_response.encoding = 'utf-8'  # fix character encoding, same fix as before

# Parse the book's page HTML
book_soup = BeautifulSoup(book_response.text, "html.parser")

# The breadcrumb navigation looks like: Home / Books / Travel / book title
# We want the 3rd link, which is the category (Home=0, Books=1, Category=2)
breadcrumb = book_soup.find("ul", class_="breadcrumb")
category = breadcrumb.find_all("a")[2].text

print(category)  # should print the genre, e.g. "Travel" or "Poetry"

Poetry


In [26]:
#the complete loop:
import time

all_books_full = []

for book in books:
    # --- Basic info from the homepage ---
    title = book.h3.a["title"]
    price_text = book.find("p", class_="price_color").text
    price_gbp = float(price_text.replace("£", ""))
    price_inr = round(price_gbp * conversion_rate, 2)

    rating_class = book.find("p", class_="star-rating")["class"]
    rating_word = rating_class[1]
    rating_number = rating_map[rating_word]

    # --- Visit the book's individual page (for Category AND Stock Count) ---
    book_link = "https://books.toscrape.com/" + book.h3.a["href"]
    book_response = requests.get(book_link)
    book_response.encoding = 'utf-8'
    book_soup = BeautifulSoup(book_response.text, "html.parser")

    # Category from breadcrumb
    breadcrumb = book_soup.find("ul", class_="breadcrumb")
    category = breadcrumb.find_all("a")[2].text

    # Stock count from the individual page (this page has the real number)
    availability_text = book_soup.find("p", class_="instock availability").text
    stock_match = re.search(r'\((\d+) available\)', availability_text)
    stock_count = int(stock_match.group(1)) if stock_match else None

    all_books_full.append({
        "Title": title,
        "Price (GBP)": price_gbp,
        "Price (INR)": price_inr,
        "Rating": rating_number,
        "Stock Count": stock_count,
        "Category": category
    })

    time.sleep(0.3)

print(len(all_books_full), "books fully collected")
print(all_books_full[0])

20 books fully collected
{'Title': 'A Light in the Attic', 'Price (GBP)': 51.77, 'Price (INR)': 5435.85, 'Rating': 3, 'Stock Count': 22, 'Category': 'Poetry'}


In [27]:
#convert to a DataFrame (table) and preview it:
import pandas as pd

df = pd.DataFrame(all_books_full)
df.head()  # shows first 5 rows as a table

,Title,Price (GBP),Price (INR),Rating,Stock Count,Category
0,A Light in the Attic,51.77,5435.85,3,22,Poetry
1,Tipping the Velvet,53.74,5642.70,1,20,Historical Fiction
2,Soumission,50.10,5260.50,1,20,Fiction
3,Sharp Objects,47.82,5021.10,4,20,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5694.15,5,20,History


In [28]:
#save it as a CSV file (this is your Task 1 deliverable):
df.to_csv("books_data.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [30]:
#scrape multiple pages:
import time
from urllib.parse import urljoin  # correctly resolves relative links

all_books_full = []

for page_num in range(1, 6):  # pages 1 to 5 (100 books total)
    if page_num == 1:
        page_url = "https://books.toscrape.com/"
    else:
        page_url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"

    response = requests.get(page_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price_text = book.find("p", class_="price_color").text
        price_gbp = float(price_text.replace("£", ""))
        price_inr = round(price_gbp * conversion_rate, 2)

        rating_word = book.find("p", class_="star-rating")["class"][1]
        rating_number = rating_map[rating_word]

        # urljoin correctly builds the full URL no matter what page we're on
        book_link = urljoin(page_url, book.h3.a["href"])
        book_response = requests.get(book_link)
        book_response.encoding = 'utf-8'
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        breadcrumb = book_soup.find("ul", class_="breadcrumb")
        category = breadcrumb.find_all("a")[2].text if breadcrumb else None

        availability_text = book_soup.find("p", class_="instock availability").text
        stock_match = re.search(r'\((\d+) available\)', availability_text)
        stock_count = int(stock_match.group(1)) if stock_match else None

        all_books_full.append({
            "Title": title,
            "Price (GBP)": price_gbp,
            "Price (INR)": price_inr,
            "Rating": rating_number,
            "Stock Count": stock_count,
            "Category": category
        })

        time.sleep(0.2)

    print(f"Page {page_num} done — total books so far: {len(all_books_full)}")

print("FINISHED:", len(all_books_full), "books collected")

Page 1 done — total books so far: 20
Page 2 done — total books so far: 40
Page 3 done — total books so far: 60
Page 4 done — total books so far: 80
Page 5 done — total books so far: 100
FINISHED: 100 books collected


In [32]:
df = pd.DataFrame(all_books_full)
df.to_csv("books_data.csv", index=False)
df.head()

,Title,Price (GBP),Price (INR),Rating,Stock Count,Category
0,A Light in the Attic,51.77,5435.85,3,22,Poetry
1,Tipping the Velvet,53.74,5642.70,1,20,Historical Fiction
2,Soumission,50.10,5260.50,1,20,Fiction
3,Sharp Objects,47.82,5021.10,4,20,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5694.15,5,20,History
